# 4. Sentiment analysis

In [1]:
# ! pip install torch

In [2]:
import nltk
import tensorflow as tf
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.models import Sequential
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
import pandas as pd
import json
import pandas as pd
import numpy as np
import plotly as plt
import warnings
warnings.filterwarnings('ignore')
pd.options.plotting.backend = "plotly"
# import yfinance as yf

## Step1&2: get the Processed data

In [4]:
topic=pd.read_excel("topic modeling in processed data.xlsx")
topic.head(2)

,publish time,headline_text,Sentence_length,doc,words,words_no_stop,words_final,words_frequency,Topic,7,1,9,6,3
0,2023-01-02,Hedge Fund Align Urges Korean Banks to Boost S...,64,hedge fund align urges korean banks to boost s...,"['hedge', 'fund', 'align', 'urges', 'korean', ...","['hedge', 'fund', 'align', 'urges', 'korean', ...","['hedg', 'fund', 'align', 'urg', 'korean', 'ba...","[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1...",3,0,0,0,0,1
1,2023-01-02,Israel’s Smotrich May Adopt Rules Forcing Disc...,63,israels smotrich may adopt rules forcing disco...,"['israels', 'smotrich', 'may', 'adopt', 'rules...","['israels', 'smotrich', 'may', 'adopt', 'rules...","['israel', 'smotrich', 'may', 'adopt', 'rule',...","[(9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (...",3,0,0,0,0,1


In [5]:
fin_svb=pd.read_csv(".\data source\SIVBQ_20230102_20230331.csv")
fin_svb['Date']=pd.to_datetime(fin_svb['Date'])
fin_svb.set_index('Date',inplace=True)
fin_svb.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2023-01-03,232.169998,235.000000,221.649994,225.220001,225.220001,764645
2023-01-04,230.100006,241.632507,228.710007,240.059998,240.059998,854687
2023-01-05,235.710007,237.389999,222.410004,232.589996,232.589996,985262
2023-01-06,237.119995,248.190002,231.429993,245.789993,245.789993,1017532
2023-01-09,247.960007,254.940002,244.529999,249.429993,249.429993,1102929


In [6]:
data=pd.read_excel("news_titles_with_attitude.xlsx")
data.head(2)

,publish time,headline_text,Sentence_length,doc,words,words_no_stop,words_final,words_frequency,attitude
0,2023-01-02,Hedge Fund Align Urges Korean Banks to Boost S...,64,hedge fund align urges korean banks to boost s...,"['hedge', 'fund', 'align', 'urges', 'korean', ...","['hedge', 'fund', 'align', 'urges', 'korean', ...","['hedg', 'fund', 'align', 'urg', 'korean', 'ba...","[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1...",0.0
1,2023-01-02,Israel’s Smotrich May Adopt Rules Forcing Disc...,63,israels smotrich may adopt rules forcing disco...,"['israels', 'smotrich', 'may', 'adopt', 'rules...","['israels', 'smotrich', 'may', 'adopt', 'rules...","['israel', 'smotrich', 'may', 'adopt', 'rule',...","[(9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (...",0.0


In [7]:
dt = data[ ['publish time','Sentence_length','headline_text','words_final','attitude'] ]
dt['Topic'] = topic[ ['Topic'] ]
dt.head(2)

,publish time,Sentence_length,headline_text,words_final,attitude,Topic
0,2023-01-02,64,Hedge Fund Align Urges Korean Banks to Boost S...,"['hedg', 'fund', 'align', 'urg', 'korean', 'ba...",0.0,3
1,2023-01-02,63,Israel’s Smotrich May Adopt Rules Forcing Disc...,"['israel', 'smotrich', 'may', 'adopt', 'rule',...",0.0,3


In [8]:
dt [ dt['attitude']>0 ].count()

publish time       1335
Sentence_length    1335
headline_text      1335
words_final        1335
attitude           1335
Topic              1335
dtype: int64

In [9]:
dt [ dt['attitude']==0 ].count()

publish time       6021
Sentence_length    6021
headline_text      6021
words_final        6021
attitude           6021
Topic              6021
dtype: int64

In [10]:
svb_dates = fin_svb.index
all_dates = dt['publish time'].unique()

m=[]
for d in all_dates:
    if d in svb_dates:
        m.append( round(fin_svb.Close.loc[d], 2) )
    else:
        m.append(np.nan)

svb = pd.DataFrame(index=all_dates)
# svb['date'] = all_dates

svb['Close'] = m
svb.fillna(axis=0, method="ffill",inplace=True)
print(svb)
svb.plot(
    kind='line',
    title='SVB stock price',
)

             Close
2023-01-02     NaN
2023-01-03  225.22
2023-01-04  240.06
2023-01-05  232.59
2023-01-06  245.79
...            ...
2023-03-27  106.04
2023-03-28    0.40
2023-03-29    0.97
2023-03-30    0.89
2023-03-31    0.89

[88 rows x 1 columns]


In [11]:
svb_dates = svb.index
all_dates = dt['publish time'].tolist()

m=[]
for d in all_dates:
    if d in svb_dates:
        m.append( round(svb.Close.loc[d], 2) )
    else:
        m.append(np.nan)

dt['svb_close_price'] = m
dt.loc[20:40]

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price
20,2023-01-03,52,Citizens Financial Cut to Neutral at Wedbush; ...,"['citizen', 'financi', 'cut', 'neutral', 'wedb...",0.000000,6,225.22
21,2023-01-03,62,Bank of China A Shares Raised to Add at CGS-CI...,"['bank', 'china', 'share', 'rais', 'add', 'cgs...",0.000000,3,225.22
22,2023-01-03,52,MANDATE: BNG Bank EUR Benchmark 10Y Social Bon...,"['mandat', 'bng', 'bank', 'eur', 'benchmark', ...",0.033333,9,225.22
23,2023-01-03,59,Citic Bank A Shares Raised to Add at CGS-CIMB;...,"['citic', 'bank', 'share', 'rais', 'add', 'cgs...",0.000000,6,225.22
24,2023-01-03,52,Bocom H Shares Raised to Add at CGS-CIMB; PT H...,"['bocom', 'h', 'share', 'rais', 'add', 'cgscim...",0.000000,6,225.22
25,2023-01-03,54,Bocom A Shares Raised to Add at CGS-CIMB; PT 5...,"['bocom', 'share', 'rais', 'add', 'cgscimb', '...",0.000000,6,225.22
26,2023-01-03,49,M&T Bank Raised to Outperform at Wedbush; PT $170,"['mt', 'bank', 'rais', 'outperform', 'wedbush'...",0.000000,6,225.22
27,2023-01-03,59,Romania Dec. International Reserves Rise to EU...,"['romania', 'dec', 'intern', 'reserv', 'rise',...",0.000000,2,225.22
28,2023-01-03,63,"Credit Suisse’s Cathal Deasy to Leave Bank, Fi...","['credit', 'suiss', 'cathal', 'deasi', 'leav',...",0.000000,2,225.22
29,2023-01-03,50,*MANDATE: WESTPAC £BENCHMARK 5Y SONIA COVERED ...,"['mandat', 'westpac', 'benchmark', 'sonia', 'c...",0.000000,9,225.22


## Step 3: Feature extraction

### Word2Vec

In [12]:
dt["sentiment"]=[[1,0] if x>0 else [0,1] for x in dt["attitude"]]

In [13]:
# choose headlines when headline exist
dt=dt[dt["Sentence_length"]>0]

In [14]:
from gensim.models import Word2Vec
model = Word2Vec(dt['words_final'], vector_size=50, window=2, min_count=1, sg=0)
w2v_weights=model.wv.vectors
vocab_size, embedding_size = w2v_weights.shape
print("Vocabulary Size: {} - Embedding Dim: {}".format(vocab_size, embedding_size))


Vocabulary Size: 37 - Embedding Dim: 50


In [15]:
xreview=[]
for l in dt.index:
        doc=[w  for w in dt.loc[l,"words_final"]  if w in model.wv.index_to_key]
        if len(doc)>0:
            xreview.append(model.wv[doc])
        else:
            print(l,doc)
            xreview.append(np.zeros(model.vector_size))
dt["w2v"]=xreview

check

In [16]:
dt.head(2)

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v
0,2023-01-02,64,Hedge Fund Align Urges Korean Banks to Boost S...,"['hedg', 'fund', 'align', 'urg', 'korean', 'ba...",0.0,3,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864..."
1,2023-01-02,63,Israel’s Smotrich May Adopt Rules Forcing Disc...,"['israel', 'smotrich', 'may', 'adopt', 'rule',...",0.0,3,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864..."


In [17]:
max_length = 0
min_length=10000
for row in dt['w2v']:
  if len(row) < min_length:
        min_length = len(row)
  if len(row) > max_length:
        max_length = len(row)
print(min_length,max_length)

23 95


check

In [18]:
#for short sentences, we prepend zero padding so all input to RNN has same length

padded_w2v = []
for v in dt["w2v"]:
    zero_padding_cnt = max_length - v.shape[0]
    pad = np.zeros((1, 50))
    for i in range(zero_padding_cnt):
        v = np.concatenate((pad, v), axis=0)
    padded_w2v.append(v)

dt["padded_w2v"]=padded_w2v 

In [19]:
dt.head(2)

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v,padded_w2v
0,2023-01-02,64,Hedge Fund Align Urges Korean Banks to Boost S...,"['hedg', 'fund', 'align', 'urg', 'korean', 'ba...",0.0,3,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,2023-01-02,63,Israel’s Smotrich May Adopt Rules Forcing Disc...,"['israel', 'smotrich', 'may', 'adopt', 'rule',...",0.0,3,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


## Step 4: Spilt dataset into train and test dataset

In [20]:
from torch import randperm

np.random.seed(42) # Set the random seed to 42

len(dt)
meaningless_number = randperm(len(dt)).tolist() # 生成乱序的索引
dt['meaningless_number'] = meaningless_number
dt.sort_values(by='meaningless_number',inplace=True)
dt

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v,padded_w2v,meaningless_number
2572,2023-02-02,62,Santander Polska Expects Net Interest Margin t...,"['santand', 'polska', 'expect', 'net', 'intere...",0.0,1,333.50,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",0
7772,2023-03-29,55,Dutch Trade Union Plans Another Strike at ING ...,"['dutch', 'trade', 'union', 'plan', 'anoth', '...",0.0,0,0.97,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",1
2286,2023-01-30,62,MANDATE: Arbejdernes Landsbank SEK and/or NOK ...,"['mandat', 'arbejdern', 'landsbank', 'sek', 'a...",0.0,9,293.96,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",2
5566,2023-03-10,63,ASIC: ANZ Penalized A$10M Over Home Loan Intro...,"['asic', 'anz', 'penal', 'home', 'loan', 'intr...",0.0,3,106.04,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",3
5809,2023-03-13,56,Banks’ Plunges and Trading Halts Show Crisis N...,"['bank', 'plung', 'trade', 'halt', 'show', 'cr...",0.0,7,106.04,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",4
...,...,...,...,...,...,...,...,...,...,...,...
4978,2023-03-03,64,Hungary Central Bank Dismisses Bank Plea to Sc...,"['hungari', 'central', 'bank', 'dismiss', 'ban...",0.0,3,284.41,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",8132
3180,2023-02-08,64,Mauritius’s SBM Names Nilamber as Group Chief ...,"['mauritiuss', 'sbm', 'name', 'nilamb', 'group...",0.0,4,320.40,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",8133
4188,2023-02-21,50,Mizuho Trust Fires Employee For Embezzling 60m...,"['mizuho', 'trust', 'fire', 'employe', 'embezz...",0.0,9,285.71,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",8134
3126,2023-02-08,64,Goldman Sachs Names Banker in Senior Americas ...,"['goldman', 'sach', 'name', 'banker', 'senior'...",0.0,6,320.40,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",8135


In [21]:
# spilt the df:dt into train an dtest dataset
train=dt.iloc[:int(0.7*len(dt))]
test=dt.iloc[int(0.7*len(dt)):]

train.sort_values(by='publish time', ascending=True, inplace=True)
train.drop(columns='meaningless_number', inplace=True)
test.sort_values(by='publish time', ascending=True, inplace=True)
test.drop(columns='meaningless_number', inplace=True)


In [22]:
dt.sort_values(by='publish time', ascending=True, inplace=True)
dt.drop(columns='meaningless_number', inplace=True)

check

In [23]:
dt

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v,padded_w2v
8,2023-01-02,45,"MANDATE: BPCE EUR Benchmark SNP; Long 5Y, 10Y","['mandat', 'bpce', 'eur', 'benchmark', 'snp', ...",-0.050000,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,2023-01-02,61,Actis Is Said to Be in Talks to Refinance Min....,"['acti', 'said', 'talk', 'refin', 'min', 'inrb...",0.000000,0,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
14,2023-01-02,60,*GUIDANCE: ERSTE GROUP BANK €BENCHMARK 6Y COVE...,"['guidanc', 'erst', 'group', 'bank', 'benchmar...",0.000000,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
13,2023-01-02,51,GUIDANCE: Erste EUR Benchmark 6Y Covered MS+23...,"['guidanc', 'erst', 'eur', 'benchmark', 'cover...",0.000000,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
6,2023-01-02,64,*NEW DEAL:EDELWEISS FINANCIAL PLANS UP TO INR4...,"['new', 'dealedelweiss', 'financi', 'plan', 'i...",0.045455,0,NaN,"[1, 0]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...,...,...,...,...,...
8012,2023-03-31,63,UAE Cenbank Revokes the Licence of MTS Bank Br...,"['uae', 'cenbank', 'revok', 'licenc', 'mt', 'b...",0.000000,7,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8030,2023-03-31,61,"Fed Rate Cut Would Lead to Stagflation, Instab...","['fed', 'rate', 'cut', 'lead', 'stagflat', 'in...",0.000000,7,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8079,2023-03-31,38,Poland’s mBank Appoints Ruhland as CFO,"['poland', 'mbank', 'appoint', 'ruhland', 'cfo']",0.000000,7,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8041,2023-03-31,63,US LEVLOANS CALENDAR: Melissa & Doug’s A&E Ter...,"['us', 'levloan', 'calendar', 'melissa', 'doug...",-0.125000,7,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [24]:
test

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v,padded_w2v
5,2023-01-02,64,NEW DEAL: Edelweiss Financial Plans Up to INR4...,"['new', 'deal', 'edelweiss', 'financi', 'plan'...",0.045455,0,NaN,"[1, 0]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
15,2023-01-02,56,Paschi Says It Has Overcome Risks to Business ...,"['paschi', 'say', 'overcom', 'risk', 'busi', '...",0.000000,7,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
6,2023-01-02,64,*NEW DEAL:EDELWEISS FINANCIAL PLANS UP TO INR4...,"['new', 'dealedelweiss', 'financi', 'plan', 'i...",0.045455,0,NaN,"[1, 0]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
11,2023-01-02,59,*MANDATE: BERLIN HYP €1B COVERED; LONG 3Y SOCI...,"['mandat', 'berlin', 'hyp', 'b', 'cover', 'lon...",-0.072222,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,2023-01-02,61,Actis Is Said to Be in Talks to Refinance Min....,"['acti', 'said', 'talk', 'refin', 'min', 'inrb...",0.000000,0,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...,...,...,...,...,...
8125,2023-03-31,60,"*AUSTRALIA FEB. CREDIT TO BUSINESS, CONSUMERS ...","['australia', 'feb', 'credit', 'busi', 'consum...",0.000000,2,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8052,2023-03-31,45,*PHILIPPINES FEB. BANK NET LOANS RISE 10% Y/Y,"['philippin', 'feb', 'bank', 'net', 'loan', 'r...",0.000000,2,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8130,2023-03-31,59,"JBIC, Others Give Vietcombank $300m Loan for G...","['jbic', 'other', 'give', 'vietcombank', 'm', ...",-0.200000,9,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8134,2023-03-31,53,*SMFG SOUNDS OUT INVESTORS ABOUT AT1 YEN BOND ...,"['smfg', 'sound', 'investor', 'yen', 'bond', '...",0.000000,5,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [25]:
train

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v,padded_w2v
12,2023-01-02,61,Paschi Jumps; Bank Says Risks to Business Cont...,"['paschi', 'jump', 'bank', 'say', 'risk', 'bus...",0.000000,7,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
10,2023-01-02,63,MANDATE: Berlin Hyp EU1b WNG Covered; Long 3Y ...,"['mandat', 'berlin', 'hyp', 'eub', 'wng', 'cov...",-0.072222,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
14,2023-01-02,60,*GUIDANCE: ERSTE GROUP BANK €BENCHMARK 6Y COVE...,"['guidanc', 'erst', 'group', 'bank', 'benchmar...",0.000000,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
0,2023-01-02,64,Hedge Fund Align Urges Korean Banks to Boost S...,"['hedg', 'fund', 'align', 'urg', 'korean', 'ba...",0.000000,3,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8,2023-01-02,45,"MANDATE: BPCE EUR Benchmark SNP; Long 5Y, 10Y","['mandat', 'bpce', 'eur', 'benchmark', 'snp', ...",-0.050000,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...,...,...,...,...,...
8039,2023-03-31,56,Dometic Renews Credit Facility Agreement With ...,"['domet', 'renew', 'credit', 'facil', 'agreeme...",0.000000,3,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8005,2023-03-31,58,Storebrand Raised to Buy at Kepler Cheuvreux; ...,"['storebrand', 'rais', 'buy', 'kepler', 'cheuv...",0.000000,6,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8058,2023-03-31,60,China Construction Bank Loan Growth to Support...,"['china', 'construct', 'bank', 'loan', 'growth...",0.000000,3,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
8089,2023-03-31,62,*COLLINS: STILL SEEING QUITE A BIT OF STRENGTH...,"['collin', 'still', 'see', 'quit', 'bit', 'str...",0.000000,4,0.89,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [26]:
test["sentiment"].value_counts()

sentiment
[0, 1]    2033
[1, 0]     409
Name: count, dtype: int64

In [27]:
train["sentiment"].value_counts()

sentiment
[0, 1]    4769
[1, 0]     926
Name: count, dtype: int64

In [28]:
X_train=np.array(train["padded_w2v"].tolist())
X_test=np.array(test["padded_w2v"].tolist())
Y_train=np.array(train["sentiment"].tolist())
Y_test=np.array(test["sentiment"].tolist())

## Step 5:  Modeling

In [29]:
# LSTM model
model = Sequential()
model.add(LSTM(32))
model.add(Dense(2, activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [30]:
print('Train...')
model.fit(X_train, Y_train,epochs=10)

Train...
Epoch 1/10
178/178 [==============================] - 8s 29ms/step - loss: 0.4605 - accuracy: 0.8195
Epoch 2/10
178/178 [==============================] - 5s 29ms/step - loss: 0.4434 - accuracy: 0.8374
Epoch 3/10
178/178 [==============================] - 6s 31ms/step - loss: 0.4421 - accuracy: 0.8374
Epoch 4/10
178/178 [==============================] - 5s 28ms/step - loss: 0.4396 - accuracy: 0.8374
Epoch 5/10
178/178 [==============================] - 5s 27ms/step - loss: 0.4347 - accuracy: 0.8374
Epoch 6/10
178/178 [==============================] - 5s 28ms/step - loss: 0.4319 - accuracy: 0.8378
Epoch 7/10
178/178 [==============================] - 5s 27ms/step - loss: 0.4286 - accuracy: 0.8390
Epoch 8/10
178/178 [==============================] - 5s 27ms/step - loss: 0.4283 - accuracy: 0.8409
Epoch 9/10
178/178 [==============================] - 5s 30ms/step - loss: 0.4231 - accuracy: 0.8435
Epoch 10/10
178/178 [==============================] - 5s 29ms/step - loss: 0.4205

In [31]:
score, acc = model.evaluate(X_train, Y_train, verbose=2)
print('Train score:', score)# Loss value
print('Train accuracy:', acc)

178/178 - 2s - loss: 0.4181 - accuracy: 0.8451 - 2s/epoch - 13ms/step
Train score: 0.41813361644744873
Train accuracy: 0.845127284526825


In [32]:
score, acc = model.evaluate(X_test, Y_test, verbose=2)
print('Test score:', score)
print('Test accuracy:', acc)

77/77 - 1s - loss: 0.4404 - accuracy: 0.8370 - 867ms/epoch - 11ms/step
Test score: 0.44037100672721863
Test accuracy: 0.8370188474655151


In [33]:
temp=model.predict(X_test)
test["pred"]=[1 if x[0]>x[1] else 0   for x in list(temp)]

77/77 [==============================] - 1s 11ms/step


In [34]:
temp=model.predict(X_test)
test["pred"]=[1 if x[0]>x[1] else 0   for x in list(temp)]
confusion_matrix=pd.DataFrame()
confusion_matrix["realY"]=[1 if x[0]==1 else 0 for x in test["sentiment"].values]
confusion_matrix["predictY"]=test["pred"].values
pd.crosstab(confusion_matrix["predictY"],confusion_matrix["realY"])

77/77 [==============================] - 1s 13ms/step


realY,0,1
predictY,,
0,2032,397
1,1,12


In [35]:
test["prob1"]=[x[0]  for x in list(temp)]
test["prob_fast"]=(test["prob1"]).rolling(100).mean()
test["prob_slow"]=(test["prob1"]).rolling(50).mean()

check

## Step 6: Model result

In [36]:
train.head(2)

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v,padded_w2v
12,2023-01-02,61,Paschi Jumps; Bank Says Risks to Business Cont...,"['paschi', 'jump', 'bank', 'say', 'risk', 'bus...",0.000000,7,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
10,2023-01-02,63,MANDATE: Berlin Hyp EU1b WNG Covered; Long 3Y ...,"['mandat', 'berlin', 'hyp', 'eub', 'wng', 'cov...",-0.072222,9,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [37]:
test.head(2)

,publish time,Sentence_length,headline_text,words_final,attitude,Topic,svb_close_price,sentiment,w2v,padded_w2v,pred,prob1,prob_fast,prob_slow
5,2023-01-02,64,NEW DEAL: Edelweiss Financial Plans Up to INR4...,"['new', 'deal', 'edelweiss', 'financi', 'plan'...",0.045455,0,NaN,"[1, 0]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",0,0.200375,NaN,NaN
15,2023-01-02,56,Paschi Says It Has Overcome Risks to Business ...,"['paschi', 'say', 'overcom', 'risk', 'busi', '...",0.000000,7,NaN,"[0, 1]","[[-0.23918201, 0.57706356, -0.39274916, 0.0864...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",0,0.213150,NaN,NaN


### Results statistics in test dataset

In [38]:
test_gp = test.groupby(['publish time']).agg({
    'headline_text':[len],
    'pred':[np.mean,np.max],
    'prob1': [np.mean],
    'prob_fast': [np.mean],
    'prob_slow': [np.mean],
    })
test_gp.head(20)

headline_text      pred          prob1 prob_fast prob_slow
                       len      mean amax      mean      mean      mean
publish time                                                           
2023-01-02              10  0.000000    0  0.168908       NaN       NaN
2023-01-03              24  0.000000    0  0.110893       NaN       NaN
2023-01-04              37  0.000000    0  0.104767       NaN  0.107124
2023-01-05              27  0.000000    0  0.145302       NaN  0.116920
2023-01-06              14  0.000000    0  0.153300  0.123575  0.137201
2023-01-07               1  0.000000    0  0.235837  0.124698  0.146507
2023-01-09              35  0.000000    0  0.117893  0.125639  0.137293
2023-01-10              33  0.000000    0  0.107437  0.127695  0.114818
2023-01-11              36  0.027778    1  0.142023  0.123428  0.122617
2023-01-12              23  0.000000    0  0.124029  0.125480  0.138374
2023-01-13              42  0.000000    0  0.121159  0.128482  0.127042
2023-01-14              10  0.000000    0  0.122359  0.128164  0.119608
2023-01-16              28  0.000000    0  0.141387  0.126174  0.123884
2023-01-17              31  0.000000    0  0.125272  0.126595  0.131901
2023-01-18              35  0.000000    0  0.131071  0.129990  0.132549
2023-01-19              42  0.023810    1  0.119434  0.132194  0.131228
2023-01-20              41  0.000000    0  0.123373  0.125551  0.117652
2023-01-21               8  0.000000    0  0.099466  0.121211  0.120680
2023-01-22               1  0.000000    0  0.130754  0.119229  0.119696
2023-01-23              34  0.000000    0  0.138618  0.120086  0.123054

In [39]:
test_gp_hd_len =  test_gp['headline_text']
test_gp_hd_len

,len
publish time,
2023-01-02,10
2023-01-03,24
2023-01-04,37
2023-01-05,27
2023-01-06,14
...,...
2023-03-27,40
2023-03-28,50
2023-03-29,43


In [40]:
test_gp_prob_fast = test_gp['prob_fast']
test_gp_prob_fast

,mean
publish time,
2023-01-02,NaN
2023-01-03,NaN
2023-01-04,NaN
2023-01-05,NaN
2023-01-06,0.123575
...,...
2023-03-27,0.145289
2023-03-28,0.146472
2023-03-29,0.139793


### Result visualization in test dataset

In [41]:
svb.plot(
    kind='line',
    title='SVB stock price',
)

In [42]:
# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
r=4
# Add traces
fig.add_trace(
    go.Scatter(x=test['publish time'], y=test.svb_close_price, name='SVB stock price', mode='lines'),secondary_y=False)

fig.add_trace(
    go.Scatter(x=test_gp_prob_fast.index, y=test_gp_prob_fast['mean'], name='Prob_fast(50) of Positive Sentiment', mode='lines',  line=dict(color='red')),secondary_y=True)


# Add figure title
fig.update_layout(
    title_text="Trending of Positive Sentiment ",
    width=1200,
    height=600,
)

# Set x-axis title
fig.update_xaxes(title_text="Date")

# Set y-axes titles
fig.update_yaxes(title_text="SVB stock price ", secondary_y=False)
fig.update_yaxes(title_text="<b>Prob of Positive Sentiment</b>", secondary_y=True)

fig.show()

In [43]:
# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
r=4
# Add traces
fig.add_trace(
    go.Scatter(x=test['publish time'], y=test['svb_close_price'], name='SVB stock price', mode='lines'),secondary_y=False)

fig.add_trace(
    go.Bar(x=test_gp_hd_len.index, y=test_gp_hd_len['len'], name='the number of daily headline'),secondary_y=True)


# Add figure title
fig.update_layout(
    title_text="Trending of Positive Sentiment ",
    width=1200,
    height=600,
)

# Set x-axis title
fig.update_xaxes(title_text="Date")

# Set y-axes titles
fig.update_yaxes(title_text="SVB stock price ", secondary_y=False)
fig.update_yaxes(title_text="<b>Prob of Positive Sentiment</b>", secondary_y=True)

fig.show()

# 5. Sentiment result and Topic modeling result

## topic modeling result

In [79]:
def count_number_of_headline_within_top_details(top_n_index=0,top_details_name=0,top_details=0):
    a = top_details_name
    b = (top_n_index+1)
    c = top_details
    print('the headine number with topic',a,'ranked#',b,':',c)


top5=dt['Topic'].value_counts().index[0:5]

top_details = dt['Topic'].value_counts().tolist()
len_top_details = len(top_details)
top_details_name = dt['Topic'].value_counts().index.tolist()

In [80]:
print(len_top_details,'topics are generated here')
print('------------------------------------------')
print(top_details)
print(top_details_name)
print('------------------------------------------')
for i in range(len_top_details):
    count_number_of_headline_within_top_details(i,top_details_name[i],top_details[i])

10 topics are generated here
------------------------------------------
[1210, 1169, 1089, 1035, 750, 742, 604, 567, 505, 466]
[7, 1, 9, 6, 3, 0, 5, 2, 4, 8]
------------------------------------------
the headine number with topic 7 ranked# 1 : 1210
the headine number with topic 1 ranked# 2 : 1169
the headine number with topic 9 ranked# 3 : 1089
the headine number with topic 6 ranked# 4 : 1035
the headine number with topic 3 ranked# 5 : 750
the headine number with topic 0 ranked# 6 : 742
the headine number with topic 5 ranked# 7 : 604
the headine number with topic 2 ranked# 8 : 567
the headine number with topic 4 ranked# 9 : 505
the headine number with topic 8 ranked# 10 : 466


## <b> topic=topic7 </b>

### Results statistics in test dataset

In [122]:
test1 = test[test['Topic']==top5[0]]

In [123]:
test_gp1 = test1.groupby(['publish time']).agg({
    'headline_text':[len],
    'pred':[np.mean,np.max],
    'prob1': [np.mean],
    'prob_fast': [np.mean],
    'prob_slow': [np.mean],
    })
test_gp1.head()

headline_text pred          prob1 prob_fast prob_slow
                       len mean amax      mean      mean      mean
publish time                                                      
2023-01-02               1  0.0    0  0.213150       NaN       NaN
2023-01-03               1  0.0    0  0.131326       NaN       NaN
2023-01-04               3  0.0    0  0.199756       NaN  0.106729
2023-01-05               4  0.0    0  0.167288       NaN  0.118911
2023-01-06               2  0.0    0  0.160661  0.123647  0.138895

In [124]:
test_gp1_hd_len =  test_gp1['headline_text']
test_gp1_hd_len

,len
publish time,
2023-01-02,1
2023-01-03,1
2023-01-04,3
2023-01-05,4
2023-01-06,2
...,...
2023-03-27,7
2023-03-28,10
2023-03-29,8


In [125]:
test_gp1_prob_fast = test_gp1['prob_fast']
test_gp1_prob_fast

,mean
publish time,
2023-01-02,NaN
2023-01-03,NaN
2023-01-04,NaN
2023-01-05,NaN
2023-01-06,0.123647
...,...
2023-03-27,0.145485
2023-03-28,0.146278
2023-03-29,0.140006


### Result visualization in test dataset

In [126]:
i=0
count_number_of_headline_within_top_details(i,top_details_name[i],top_details[i])

the headine number with topic 7 ranked# 1 : 1210


In [127]:

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
r=4
# Add traces
fig.add_trace(
    go.Scatter(x=test['publish time'], y=test.svb_close_price, name='SVB stock price', mode='lines'),secondary_y=False)

fig.add_trace(
    go.Scatter(x=test_gp1_prob_fast.index, y=test_gp1_prob_fast['mean'], name='Prob_fast(50) of Positive Sentiment', mode='lines',  line=dict(color='red')),secondary_y=True)


# Add figure title
fig.update_layout(
    title_text="Trending of Positive Sentiment ",
    width=1200,
    height=600,
)

# Set x-axis title
fig.update_xaxes(title_text="Date")

# Set y-axes titles
fig.update_yaxes(title_text="SVB stock price ", secondary_y=False)
fig.update_yaxes(title_text="<b>Prob of Positive Sentiment</b>", secondary_y=True)

fig.show()

In [128]:

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
r=4
# Add traces
fig.add_trace(
    go.Scatter(x=test['publish time'], y=test.svb_close_price, name='SVB stock price', mode='lines'),secondary_y=False)

fig.add_trace(
    go.Bar(x=test_gp1_hd_len.index, y=test_gp1_hd_len['len'], name='the number of daily headline'),secondary_y=True)


# Add figure title
fig.update_layout(
    title_text="Trending of Positive Sentiment ",
    width=1200,
    height=600,
)

# Set x-axis title
fig.update_xaxes(title_text="Date")

# Set y-axes titles
fig.update_yaxes(title_text="SVB stock price ", secondary_y=False)
fig.update_yaxes(title_text="<b>Prob of Positive Sentiment</b>", secondary_y=True)


---

## Experiment: Result visualization in whole dataset

In [129]:
wdt=dt
# wdt

w_X=np.array(wdt["padded_w2v"].tolist())

w_temp=model.predict(w_X)
wdt["pred"]=[1 if x[0]>x[1] else 0   for x in list(w_temp)]

wdt["prob1"]=[x[0]  for x in list(w_temp)]
wdt["prob_fast"]=(wdt["prob1"]).rolling(100).mean()
wdt["prob_slow"]=(wdt["prob1"]).rolling(50).mean()

wdt_gp = wdt.groupby(['publish time']).agg({
    'headline_text':[len],
    'pred':[np.mean,np.max,np.min],
    'prob1': [np.mean],
    'prob_fast': [np.mean],
    'prob_slow': [np.mean],
    })
wdt_gp.head(10)

255/255 [==============================] - 3s 11ms/step


headline_text      pred               prob1 prob_fast prob_slow
                       len      mean amax amin      mean      mean      mean
publish time                                                                
2023-01-02              19  0.000000    0    0  0.156137       NaN       NaN
2023-01-03              99  0.000000    0    0  0.118204  0.122947  0.121389
2023-01-04             124  0.000000    0    0  0.111773  0.114810  0.113830
2023-01-05              78  0.000000    0    0  0.136503  0.120738  0.127780
2023-01-06              62  0.000000    0    0  0.135769  0.133506  0.133842
2023-01-07               7  0.000000    0    0  0.150160  0.134444  0.135891
2023-01-08               1  0.000000    0    0  0.151530  0.137729  0.137674
2023-01-09             104  0.000000    0    0  0.112700  0.124869  0.119238
2023-01-10             121  0.008264    1    0  0.116581  0.114041  0.115562
2023-01-11             101  0.009901    1    0  0.136738  0.125306  0.131039

In [130]:
wdt_gp_hd_len =  wdt_gp['headline_text']
# wdt_gp_hd_len

In [131]:
wdt_gp_prob_fast = wdt_gp['prob_fast']
# wdt_gp_prob_fast

In [132]:
wdt_gp_prob_fast = wdt_gp['prob_fast']
# wdt_gp_prob_fast

In [133]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
r=4
# Add traces
fig.add_trace(
    go.Scatter(x=wdt['publish time'], y=wdt.svb_close_price, name='SVB stock price', mode='lines'),secondary_y=False)

fig.add_trace(
    go.Scatter(x=wdt['publish time'], y=wdt.prob_fast, name='Prob_fast(50) of Positive Sentiment', mode='lines',  line=dict(color='red')),secondary_y=True)


# Add figure title
fig.update_layout(
    title_text="Trending of Positive Sentiment ",
    width=1200,
    height=600,
)

# Set x-axis title
fig.update_xaxes(title_text="Date")

# Set y-axes titles
fig.update_yaxes(title_text="SVB stock price ", secondary_y=False)
fig.update_yaxes(title_text="<b>Prob of Positive Sentiment</b>", secondary_y=True)

fig.show()

In [134]:
# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
r=4
# Add traces
fig.add_trace(
    go.Scatter(x=wdt['publish time'], y=wdt['svb_close_price'], name='SVB stock price', mode='lines'),secondary_y=False)

fig.add_trace(
    go.Bar(x=wdt_gp_hd_len.index, y=wdt_gp_hd_len['len'], name='the number of headline'),secondary_y=True)


# Add figure title
fig.update_layout(
    title_text="Trending of Positive Sentiment ",
    width=1200,
    height=600,
)

# Set x-axis title
fig.update_xaxes(title_text="Date")

# Set y-axes titles
fig.update_yaxes(title_text="SVB stock price ", secondary_y=False)
fig.update_yaxes(title_text="<b>Prob of Positive Sentiment</b>", secondary_y=True)

fig.show()

# *Export processed data

In [137]:
wdt.to_excel('sentiment analysis result(whole processed data) .xlsx')